# LoCoMo test F1 per iteration: GRSE, GRSE w/o structure, MASkills

Per-category LoCoMo runs of `scripts/run_main.sh` (one library per category), Qwen3.8 and Gemma4 actors.
Every iteration's library is tested on the full test split with greedy decoding (`trajectories_test/iter_k`).

* **Solid line**: mean over runs. **Band** (`BAND`): `"ci"` = 95% t-interval over runs (mean ± t₀.₉₇₅,ₙ₋₁ · SD / √n), `"sd"` = mean ± 1 SD.
* **Iteration 0**: the empty library. MASkills does not test iteration 0, so its run with seed *s* reuses the iteration-0 answers of the GRSE run with seed *s* (same empty library, same questions); all curves start at the same point.
* **All**: the four category libraries' answers pooled over all test questions.
* Runs are found from the logs `logs/locomo_cat<c>_<method>_<actor>_s<seed>.out` written by `scripts/run_main.sh`.
* Outputs: `outputs/figures/locomo_iter_all*.pdf` and `outputs/figures/locomo_iter_percat*.pdf`.

In [ ]:
import glob, json, re
from pathlib import Path
from statistics import fmean, stdev
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# repository root (this notebook lives in notebooks/)
ROOT = Path.cwd().parent if (Path.cwd().parent / "GRSE").is_dir() else Path.cwd()
FIG_DIR = ROOT / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
ITERS = range(6)
CATS = {1: "Multi-hop", 2: "Temporal", 3: "Open-domain", 4: "Single-hop"}
ACTORS = {"qwen": "Qwen3.8-27B", "gemma": "Gemma4-26B-A4B"}
METHODS = {"grse": "GRSE", "grse_wo": "GRSE w/o structure", "maskills": "MASkills"}

## 1. Runs (from the logs of `scripts/run_main.sh`)

In [ ]:
LOG = re.compile(r"locomo_cat(\d)_(grse_wo|grse|maskills)_(qwen|gemma)_s(\d+)\.out$")

def find_runs():
    """{actor: {method: {cat: {seed: run directory}}}} from logs/locomo_cat*.out (smoke runs excluded)."""
    runs = {}
    for log in sorted(glob.glob(str(ROOT / "logs" / "locomo_cat*_s*.out"))):
        m = LOG.search(log)
        if not m:
            continue
        cat, method, actor, seed = int(m[1]), m[2], m[3], int(m[4])
        run_id = re.search(r"(?:GRPO|MASKILLS) run: (\S+)", Path(log).read_text(errors="ignore"))
        if not run_id:
            continue
        root = ROOT / ("experiments/runs" if method == "maskills" else "experiments/grpo/runs")
        runs.setdefault(actor, {}).setdefault(method, {}).setdefault(cat, {})[seed] = root / run_id[1]
    return runs

RUNS = find_runs()
for a in RUNS:
    for k in RUNS[a]:
        for c in sorted(RUNS[a][k]):
            print(a, METHODS[k], CATS[c], {s: d.name for s, d in sorted(RUNS[a][k][c].items())})

## 2. Per-question F1 / BLEU at every iteration

In [ ]:
def load_iter(run, it):
    """{question id: (F1, BLEU)} of one iteration's test answers ({} if not tested)."""
    out = {}
    for f in glob.glob(str(Path(run) / "trajectories_test" / f"iter_{it}" / "*.json")):
        d = json.loads(Path(f).read_text())
        task, metrics = d["task"], d["metadata"].get("qa_metrics", {})
        out[str(task.get("task_id") or task.get("question"))] = (metrics.get("f1"), metrics.get("bleu"))
    return out

# SCORES[actor][method][cat][seed][iteration] = {qid: (f1, bleu)}
# Loading every iteration's answers takes a while, so the scores are cached; the cache is
# rebuilt whenever the set of runs changes.
import pickle
CACHE = Path("locomo_iter_cache.pkl")
KEY = {a: {k: {c: {s: str(d) for s, d in RUNS[a][k][c].items()} for c in RUNS[a][k]} for k in RUNS[a]} for a in RUNS}
cached = pickle.loads(CACHE.read_bytes()) if CACHE.exists() else None
SCORES = cached["scores"] if cached and cached["key"] == KEY else {}
for a in ([] if SCORES else RUNS):
    SCORES[a] = {}
    for k in METHODS:
        SCORES[a][k] = {}
        for c in CATS:
            SCORES[a][k][c] = {}
            for s, d in RUNS[a].get(k, {}).get(c, {}).items():
                final = load_iter(d, 5)
                if final:                                     # skip unfinished runs
                    SCORES[a][k][c][s] = {it: (final if it == 5 else load_iter(d, it)) for it in ITERS}
    CACHE.write_bytes(pickle.dumps({"key": KEY, "scores": SCORES}))
# MASkills has no iteration-0 test: reuse the GRSE run's empty-library answers (same empty library and questions)
for a in SCORES:
    for c in CATS:
        for s, its in SCORES[a]["maskills"][c].items():
            if not its[0] and SCORES[a]["grse"][c]:
                g = SCORES[a]["grse"][c].get(s) or next(iter(SCORES[a]["grse"][c].values()))
                its[0] = g[0]
# one question set per actor and category, shared by every run and iteration
QS = {a: {c: set.intersection(*(set(its[it]) for k in METHODS for its in SCORES[a][k][c].values() for it in ITERS))
          for c in CATS} for a in SCORES}
for a in QS:
    print(a, {CATS[c]: len(QS[a][c]) for c in CATS})
    for k in METHODS:
        print("   ", METHODS[k], {CATS[c]: sorted(SCORES[a][k][c]) for c in CATS}, "(seeds)")

## 3. Per-run curves, mean and 95% t-interval over runs

In [ ]:
METRIC = 0          # 0 = F1, 1 = BLEU
BAND = "ci"         # "ci": 95% t-interval over runs;  "sd": mean ± 1 SD over runs
CI = 0.95

def run_curves(a, k, cat):
    """[[score at iteration 0..5] for each run]; cat 'all' pools the four categories by run index."""
    if cat == "all":
        runs = sorted(set.intersection(*(set(SCORES[a][k][c]) for c in CATS)))
        return [[100 * fmean(SCORES[a][k][c][r][it][q][METRIC] for c in CATS for q in QS[a][c]) for it in ITERS]
                for r in runs]
    return [[100 * fmean(SCORES[a][k][cat][r][it][q][METRIC] for q in QS[a][cat]) for it in ITERS]
            for r in sorted(SCORES[a][k][cat])]

def summary(curves):
    x = np.array(curves)
    m = x.mean(0)
    if len(x) < 2:
        return m, m, m, len(x)
    sd = x.std(0, ddof=1)
    h = stats.t.ppf((1 + CI) / 2, len(x) - 1) * sd / np.sqrt(len(x)) if BAND == "ci" else sd
    return m, m - h, m + h, len(x)

rows = []
for a in SCORES:
    for cat in ["all", *CATS]:
        for k in METHODS:
            m, lo, hi, n = summary(run_curves(a, k, cat))
            rows.append([a, "All" if cat == "all" else CATS[cat], METHODS[k], n] + [f"{v:.1f}" for v in m])
import pandas as pd
pd.DataFrame(rows, columns=["actor", "category", "method", "runs"] + [f"it{i}" for i in ITERS])

## 4. Figures

In [ ]:
# categorical slots 1-2 of the reference palette (blue, orange); fixed per method
COLORS = {"grse": "#2a78d6", "maskills": "#eb6834", "grse_wo": "#1baf7a"}
plt.rcParams.update({
    "font.family": "serif", "font.serif": ["Nimbus Roman", "Times New Roman", "DejaVu Serif"],
    "font.size": 8, "axes.titlesize": 8, "axes.labelsize": 8, "legend.fontsize": 7,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "axes.linewidth": 0.6,
    "axes.edgecolor": "#888888", "xtick.color": "#555555", "ytick.color": "#555555",
    "axes.spines.top": False, "axes.spines.right": False, "pdf.fonttype": 42,
})
METRIC_NAME = ["F1", "BLEU"][METRIC]

def panel(ax, a, cat, title):
    for k in ("maskills", "grse_wo", "grse"):  # GRSE drawn last (on top)
        curves = run_curves(a, k, cat)
        if not curves:
            continue
        m, lo, hi, n = summary(curves)
        ax.fill_between(ITERS, lo, hi, color=COLORS[k], alpha=0.18, linewidth=0)
        ax.plot(ITERS, m, color=COLORS[k], linewidth=1.5, marker="o", markersize=3,
                markeredgecolor="white", markeredgewidth=0.5, label=METHODS[k])
    ax.set_title(title)
    ax.set_xticks(list(ITERS))
    ax.grid(axis="y", color="#e6e6e6", linewidth=0.5)
    ax.set_axisbelow(True)

# main text: All, one panel per actor
fig, axes = plt.subplots(1, 2, figsize=(3.3, 1.75))
for ax, a in zip(axes, SCORES):
    panel(ax, a, "all", ACTORS[a])
    ax.set_xlabel("Iteration")
axes[0].set_ylabel(f"Test {METRIC_NAME} (×100)")
h, l = axes[0].get_legend_handles_labels()
fig.tight_layout(pad=0.3, w_pad=0.8, rect=(0, 0, 1, 0.86))
fig.legend(h[::-1], l[::-1], loc="upper center", ncol=len(l), frameon=False, handlelength=1.5, columnspacing=1.0, bbox_to_anchor=(0.5, 1.0), fontsize=6.5)
fig.savefig(FIG_DIR / f"locomo_iter_all{'' if METRIC == 0 else '_bleu'}{'' if BAND == 'ci' else '_sd'}.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# every category, rows = actors
fig, axes = plt.subplots(2, 4, figsize=(6.3, 3.2))
for i, a in enumerate(SCORES):
    for j, c in enumerate(CATS):
        ax = axes[i, j]
        panel(ax, a, c, f"{ACTORS[a].split('-')[0]}: {CATS[c]}")
        if i == 1:
            ax.set_xlabel("Iteration")
        if j == 0:
            ax.set_ylabel(f"Test {METRIC_NAME} (×100)")
h, l = axes[0, 0].get_legend_handles_labels()
fig.tight_layout(pad=0.3, h_pad=0.8, w_pad=0.6, rect=(0, 0, 1, 0.94))
fig.legend(h[::-1], l[::-1], loc="upper center", ncol=len(l), frameon=False, handlelength=1.8, bbox_to_anchor=(0.5, 1.0))
fig.savefig(FIG_DIR / f"locomo_iter_percat{'' if METRIC == 0 else '_bleu'}{'' if BAND == 'ci' else '_sd'}.pdf", bbox_inches="tight")
plt.show()